# TorchGeo Sampling for Semantic Segmentation

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/geoai/blob/main/docs/examples/torchgeo_sampling_segmentation.ipynb)

This notebook demonstrates GeoAI's TorchGeo-style sampling utilities for dynamic patch sampling from large GeoTIFFs. It uses the same NAIP water raster and mask dataset used in the `train_water_detection.ipynb` example, but it does not pre-export image chips.


## Install packages


In [ ]:
# %pip install geoai-py torchgeo lightning matplotlib

## Import libraries


In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

import geoai

## Download sample GeoTIFFs

The imagery and label mask are aligned GeoTIFFs hosted on Hugging Face. They are the same sample data used by the water detection training notebook.


In [ ]:
train_raster_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_train.tif"
train_masks_url = "https://huggingface.co/datasets/giswqs/geospatial/resolve/main/naip/naip_water_masks.tif"

train_raster_path = geoai.download_file(train_raster_url)
train_masks_path = geoai.download_file(train_masks_url)

train_raster_path, train_masks_path

## Inspect and visualize the inputs


In [ ]:
geoai.print_raster_info(train_raster_path, show_preview=False)
geoai.print_raster_info(train_masks_path, show_preview=False)

In [ ]:
geoai.view_raster(train_masks_path, nodata=0, basemap=train_raster_path)

## Build a TorchGeo segmentation dataset

`create_segmentation_dataset` wraps the image and mask rasters as TorchGeo `RasterDataset` objects and intersects them spatially. The resulting dataset can be queried by geographic bounding boxes from TorchGeo samplers.


In [ ]:
dataset = geoai.create_segmentation_dataset(
    train_raster_path,
    train_masks_path,
)

type(dataset), dataset.bounds, dataset.crs, dataset.res

## Create random and grid dataloaders

Random sampling is useful for training. Grid sampling is useful for deterministic scans over a region. For this short example, the validation loader also uses random sampling to keep the runtime small. For a production experiment, use spatially separate validation data or a grid sampler over a validation region.


In [ ]:
train_loader = geoai.create_geo_dataloader(
    dataset,
    sampler_type="random",
    size=256,
    length=64,
    batch_size=4,
    num_workers=0,
)

val_loader = geoai.create_geo_dataloader(
    dataset,
    sampler_type="random",
    size=256,
    length=16,
    batch_size=4,
    num_workers=0,
)

grid_loader = geoai.create_geo_dataloader(
    dataset,
    sampler_type="grid",
    size=256,
    stride=256,
    batch_size=4,
    num_workers=0,
)

len(train_loader), len(val_loader), len(grid_loader)

## Inspect a sampled batch

TorchGeo batches are dictionaries containing tensors plus geospatial metadata. `geo_sample_to_tuple` converts them to the `(image, mask)` format expected by standard segmentation training code.


In [ ]:
batch = next(iter(train_loader))
print(batch.keys())
print(batch["image"].shape, batch["image"].dtype)
print(batch["mask"].shape, batch["mask"].dtype)

images, masks = geoai.geo_sample_to_tuple(
    batch,
    num_channels=3,
    normalize=True,
)
masks = (masks > 0).long()
print(images.shape, images.dtype, images.min().item(), images.max().item())
print(masks.shape, masks.dtype, torch.unique(masks))

## Train a small semantic segmentation model

The model below is intentionally small so the notebook runs quickly on CPU. The important part is that it consumes patches sampled dynamically from the TorchGeo dataloader; no chip directory is created.


In [ ]:
class SmallSegNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, num_classes, kernel_size=1),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def prepare_batch(batch, device):
    images, masks = geoai.geo_sample_to_tuple(
        batch,
        num_channels=3,
        normalize=True,
    )
    masks = (masks > 0).long()
    return images.to(device), masks.to(device)


def mean_iou(logits, masks, num_classes=2):
    preds = logits.argmax(dim=1)
    scores = []
    for cls in range(num_classes):
        pred_cls = preds == cls
        mask_cls = masks == cls
        union = (pred_cls | mask_cls).sum()
        if union == 0:
            continue
        intersection = (pred_cls & mask_cls).sum()
        scores.append((intersection.float() / union.float()).item())
    return sum(scores) / len(scores) if scores else 0.0

In [ ]:
device = geoai.get_device()
model = SmallSegNet(in_channels=3, num_classes=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
device

In [ ]:
def run_epoch(loader, training=True):
    model.train(training)
    total_loss = 0.0
    total_iou = 0.0
    count = 0

    for batch in loader:
        images, masks = prepare_batch(batch, device)

        with torch.set_grad_enabled(training):
            logits = model(images)
            loss = criterion(logits, masks)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        total_loss += loss.item()
        total_iou += mean_iou(logits.detach(), masks)
        count += 1

    return total_loss / count, total_iou / count


history = []
best_val_loss = float("inf")

for epoch in range(3):
    train_loss, train_iou = run_epoch(train_loader, training=True)
    val_loss, val_iou = run_epoch(val_loader, training=False)

    if val_loss < best_val_loss:
        best_val_loss = val_loss

    history.append(
        {
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_iou": train_iou,
            "val_loss": val_loss,
            "val_iou": val_iou,
            "best_val_loss": best_val_loss,
        }
    )
    print(
        f"Epoch {epoch + 1}: "
        f"train_loss={train_loss:.3f}, train_iou={train_iou:.3f}, "
        f"val_loss={val_loss:.3f}, val_iou={val_iou:.3f}, "
        f"best_val_loss={best_val_loss:.3f}"
    )

## Plot performance metrics


In [ ]:
history_path = "torchgeo_manual_history.pth"
torch.save(
    {
        "train_loss": [row["train_loss"] for row in history],
        "val_loss": [row["val_loss"] for row in history],
        "val_iou": [row["val_iou"] for row in history],
        "train_iou": [row["train_iou"] for row in history],
        "best_val_loss": [row["best_val_loss"] for row in history],
    },
    history_path,
)

metrics_df = geoai.plot_performance_metrics(
    history_path=history_path,
    figsize=(10, 4),
    verbose=True,
)
metrics_df

### Why does the best score decrease?

When the monitored metric is a loss such as `val_loss`, lower values are better. A message like `New best score: 0.508` after `0.703` means validation loss improved. If you monitor an accuracy or IoU metric instead, use a maximization mode and the best score should increase.


## Visualize predictions from sampled patches


In [ ]:
model.eval()
batch = next(iter(val_loader))
images, masks = prepare_batch(batch, device)
with torch.no_grad():
    preds = model(images).argmax(dim=1)

n = min(4, images.shape[0])
fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
if n == 1:
    axes = axes[None, :]

for i in range(n):
    rgb = images[i].detach().cpu().permute(1, 2, 0).clamp(0, 1)
    axes[i, 0].imshow(rgb)
    axes[i, 0].set_title("Image")
    axes[i, 1].imshow(masks[i].detach().cpu(), cmap="Blues", vmin=0, vmax=1)
    axes[i, 1].set_title("Mask")
    axes[i, 2].imshow(preds[i].detach().cpu(), cmap="Blues", vmin=0, vmax=1)
    axes[i, 2].set_title("Prediction")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()

## Optional: use the same dataloaders with DINOv3 fine-tuning

The same TorchGeo dataloaders can be passed to `train_dinov3_segmentation`. This is commented out because it downloads DINOv3 weights and is much more compute intensive than the small model above. The checkpoint callback monitors `val_loss` with `mode="min"` by default, so the best score should decrease when training improves.


In [ ]:
# dinov3_model = geoai.train_dinov3_segmentation(
#     train_dataloader=train_loader,
#     val_dataloader=val_loader,
#     output_dir="dinov3_torchgeo_output",
#     num_classes=2,
#     num_epochs=10,
#     batch_size=4,
#     freeze_backbone=True,
#     monitor_metric="val_loss",
#     mode="min",
#     accelerator="auto",
#     devices="auto",
# )

## Summary

This workflow samples patches directly from georeferenced rasters and masks at training time. It avoids creating an intermediate chip directory while keeping compatibility with standard PyTorch training loops and GeoAI training functions that accept PyTorch dataloaders.
